# Real-Time ASL Sign Recognition

## Project overview

This notebook runs the project's webcam-based American Sign Language (ASL) sign recognition system.

The application uses **MediaPipe Holistic** to extract landmarks from webcam frames and a pretrained **GRU-based model** to predict a sign label. The live interface displays the predicted sign, a confidence score, low-confidence warnings, a movement indicator, and recent predictions.

### Recognition pipeline

1. Capture frames from the webcam.
2. Extract and normalize pose, hand, and selected facial landmarks.
3. Buffer recent frames and use the existing movement-based capture logic.
4. Pass the prepared sequence to the pretrained model.
5. Decode the predicted class into a sign label.
6. Display the prediction and confidence information.

> **Scope:** This is an isolated-sign recognition prototype for a fixed set of ASL classes. It does not reliably translate continuous signing into complete sentences.


## Before running

- Open this notebook from the **project root** (the folder containing `utils/` and `models/`).
- Select the Python interpreter from the project's virtual environment.
- Install dependencies from `requirements.txt` if needed.
- Make sure your webcam is available and the model files are in `models/`.

This notebook runs the existing model; it does **not** retrain it. Dataset-processing and training experiments are intentionally excluded.


## 1. Check the project files

This cell confirms that the notebook is running from the project root and that the required model and decoder files exist.

In [1]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "utils").is_dir():
    raise RuntimeError(
        "Project root not detected. Open this notebook from the folder "
        "containing the 'utils' and 'models' directories."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "models" / "best_model200_new.keras"
DECODER_PATH = PROJECT_ROOT / "models" / "index_to_gloss_200.json"

for path in (MODEL_PATH, DECODER_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required project file not found: {path}")

with DECODER_PATH.open("r", encoding="utf-8") as f:
    decoder = json.load(f)

if not decoder:
    raise ValueError("The label-mapping file is empty.")

print("Project root:", PROJECT_ROOT)
print("Model file:", MODEL_PATH.name)
print("Label-mapping file:", DECODER_PATH.name)
print("Labels in mapping:", len(decoder))
print("Project files found.")


Project root: c:\Users\Aryanth Venugopal\live-asl-translation
Model file: best_model200_new.keras
Label-mapping file: index_to_gloss_200.json
Labels in mapping: 209
Project files found.


## 2. Import the live recognition module

The live recognition module contains the webcam processing, landmark preprocessing, model inference, and display logic. No OpenAI API key is required for this local recognition workflow.

In [2]:
from utils import live_translation

print("Live recognition module loaded.")
print("Module location:", live_translation.__file__)


Live recognition module loaded.
Module location: c:\Users\Aryanth Venugopal\live-asl-translation\utils\live_translation.py


## 3. Start live ASL recognition

Run this cell to open the webcam and start recognition. The model and label mapping must correspond to each other.

**Controls**
- Press `q` to close the webcam window.
- Press `a` to remove the most recent prediction from the history, if supported by the current implementation.

The confidence score is the model's highest output score. A high score does not guarantee that the predicted sign is correct.

In [5]:
live_translation.start_live_feed(
    model_path=str(MODEL_PATH),
    encoder_path=str(DECODER_PATH),
    client=None
)


## Troubleshooting

- **`ImportError` for `live_translation`:** Confirm that the file is named `utils/live_translation.py`, then restart the notebook kernel and rerun the import cell.
- **Model or decoder not found:** Confirm that both files are in `models/` and that the notebook was opened from the project root.
- **Webcam does not open:** Close applications that may be using the camera and check the webcam index configured in `start_live_feed`.
- **Dependency errors:** Activate the project's virtual environment and install packages from `requirements.txt`.
- **Inconsistent predictions:** Lighting, camera angle, signing variation, landmark quality, and similarity between signs can affect predictions.

## Limitations

The model recognises only the classes represented in its output mapping. The displayed confidence score is not a calibrated guarantee of correctness. This prototype is not a professional sign-language interpreter.
